# 05 · Análisis
Caso 1 · Reformas Alzina. Este cuaderno responde a las cinco preguntas de Jordi con las tablas
`limpio_` creadas en `04_limpieza`. Antes de ejecutarlo, ejecuta `04_limpieza` entero.

1. **Rentabilidad:** ¿qué trabajos y clientes dejan más margen, y cómo ha cambiado desde que subieron los materiales?
2. **Cobros:** ¿quién debe cuánto y quién paga tarde?
3. **Caja e IVA:** ¿por qué hay beneficio pero no hay dinero, y cuándo se tensa la caja?
4. **Previsión de caja** de octubre a diciembre de 2026.
5. **Contratación:** ¿se puede permitir otro operario en 2027?

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

PROYECTO = Path(r"C:\Users\Annag\Desktop\portfolio\caso1_reformas_alzina_datos")
bd = sqlite3.connect(PROYECTO / "sql" / "alzina.db")

def sql(consulta):
    return pd.read_sql(consulta, bd)

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")

# Vista del banco con fechas e importes convertidos (es temporal: se crea cada vez)
bd.execute("DROP VIEW IF EXISTS banco")
bd.execute("""
CREATE TEMP VIEW banco AS
SELECT rowid AS fila,
       SUBSTR(fecha_operacion, 7, 4) || '-' || SUBSTR(fecha_operacion, 4, 2) || '-' || SUBSTR(fecha_operacion, 1, 2) AS fecha,
       concepto,
       CAST(REPLACE(REPLACE(importe, '.', ''), ',', '.') AS REAL) AS importe,
       CAST(REPLACE(REPLACE(saldo, '.', ''), ',', '.') AS REAL) AS saldo
FROM bruto_banco
""")

## 1. Rentabilidad de cada obra

**Margen de contribución** de una obra = lo facturado − materiales − subcontratas − mano de obra.
Es lo que deja cada obra para pagar los gastos de estructura (gerente, administrativa, alquiler,
furgonetas, gestoría…) y el beneficio. Todos los importes son sin IVA, porque el IVA no es de Jordi:
lo cobra y se lo paga a Hacienda.

**Decisiones**
- **Mano de obra** = horas de los partes × coste por hora de los operarios. El coste por hora es
  el coste total de los 4 operarios para la empresa (bruto + Seguridad Social) dividido entre
  las horas de los partes, calculado para cada año.
- **Materiales sin obra (ID 10):** 307 gastos (252.000 €, el 39% del material) no dicen a qué obra van.
  Dejarlos fuera haría que las obras parecieran mucho más rentables de lo que son. Se reparten entre
  las obras activas ese día, solo del tipo de trabajo para el que ese proveedor suele vender (Cuines
  Delta solo vende para cocinas), y en proporción a su presupuesto. Es una estimación: se aplica igual
  a todas las obras, así que las comparaciones entre tipos y clientes siguen siendo válidas.
- **Antes y después:** los materiales subieron en marzo de 2025. Cada obra se clasifica por su fecha de inicio.
- Solo se analizan las **obras terminadas** (220). Las 3 en curso aún no tienen todas sus facturas ni gastos.

In [ ]:
# --- Coste por hora de los operarios, por año: coste para la empresa / horas de los partes ---
coste_hora = sql("""
WITH coste AS (
    SELECT SUBSTR(mes, 1, 4) AS anio, SUM(coste_empresa) AS coste
    FROM limpio_nominas WHERE categoria LIKE 'Oficial%' GROUP BY anio
),
horas AS (
    SELECT SUBSTR(semana_inicio, 1, 4) AS anio, SUM(horas) AS horas
    FROM limpio_partes GROUP BY anio
)
SELECT anio, coste, horas, coste / horas AS coste_hora FROM coste JOIN horas USING (anio)
""")

# --- Materiales sin obra (ID 10): reparto entre las obras activas ese día -------------
# Solo entre obras del tipo de trabajo para el que ese proveedor suele vender
# (Cuines Delta solo vende para cocinas), en proporción a su presupuesto.
mat = sql("SELECT rowid AS id, fecha, proveedor, base, obra FROM limpio_gastos WHERE categoria = 'Materiales'")
obras = sql("SELECT codigo, tipo, fecha_inicio, COALESCE(fecha_fin, '2026-12-31') AS fecha_fin, presupuesto FROM limpio_obras")
obras["desde"] = pd.to_datetime(obras.fecha_inicio) - pd.Timedelta(days=7)
obras["hasta"] = pd.to_datetime(obras.fecha_fin)
tipos_proveedor = (mat.dropna(subset=["obra"])
                   .merge(obras[["codigo", "tipo"]], left_on="obra", right_on="codigo")
                   .groupby("proveedor").tipo.apply(set))
reparto = []
for g in mat[mat.obra.isna()].itertuples():
    dia = pd.Timestamp(g.fecha)
    activas = obras[(obras.desde <= dia) & (dia <= obras.hasta)]
    del_tipo = activas[activas.tipo.isin(tipos_proveedor.get(g.proveedor, set()))]
    if not del_tipo.empty:
        activas = del_tipo
    peso = activas.presupuesto / activas.presupuesto.sum()
    reparto += [(codigo, g.base * p) for codigo, p in zip(activas.codigo, peso)]
materiales = pd.concat([
    mat.dropna(subset=["obra"])[["obra", "base"]],
    pd.DataFrame(reparto, columns=["obra", "base"]),
]).groupby("obra").base.sum().rename("materiales")

# --- Margen de cada obra ---------------------------------------------------------------
ingresos = sql("SELECT obra, SUM(base) AS ingresos FROM limpio_facturas GROUP BY obra").set_index("obra").ingresos
subcontratas = sql("""SELECT obra, SUM(base) AS subcontratas FROM limpio_gastos
                      WHERE categoria = 'Subcontratas' GROUP BY obra""").set_index("obra").subcontratas
horas = sql("SELECT obra, SUBSTR(semana_inicio, 1, 4) AS anio, SUM(horas) AS horas FROM limpio_partes GROUP BY obra, anio")
horas = horas.merge(coste_hora[["anio", "coste_hora"]], on="anio")
mano_obra = (horas.horas * horas.coste_hora).groupby(horas.obra).sum().rename("mano_obra")
horas_obra = horas.groupby("obra").horas.sum().rename("horas")

margen = (sql("""SELECT o.codigo AS obra, o.tipo, o.fecha_inicio, o.fecha_fin, o.observaciones,
                        c.cliente, c.tipo_cliente
                 FROM limpio_obras AS o JOIN limpio_clientes AS c USING (cliente_id)""")
          .set_index("obra")
          .join([ingresos, materiales, subcontratas, mano_obra, horas_obra])
          .fillna({"materiales": 0, "subcontratas": 0, "mano_obra": 0, "horas": 0}))
margen["margen"] = margen.ingresos - margen.materiales - margen.subcontratas - margen.mano_obra
margen["margen_pct"] = margen.margen / margen.ingresos * 100
margen["periodo"] = margen.fecha_inicio.map(lambda d: "Antes (ene 2024 – feb 2025)" if d < "2025-03-01" else "Después (mar 2025 – sep 2026)")
# Solo obras terminadas: a las que están en curso aún les faltan facturas y gastos
margen = margen[margen.observaciones.fillna("") != "En curso"]

In [ ]:
# Coste por hora de los operarios para la empresa
coste_hora

In [ ]:
# Comprobación: todo el material está repartido y no se ha perdido ni duplicado nada
sql("SELECT ROUND(SUM(base), 2) AS materiales_en_gastos FROM limpio_gastos WHERE categoria = 'Materiales'") \
    .assign(materiales_repartidos=round(materiales.sum(), 2))

In [ ]:
import matplotlib.pyplot as plt

TINTA, TINTA_SUAVE, REJILLA = "#0b0b0b", "#52514e", "#e4e2dc"
AZUL, GRIS, NARANJA = "#2a78d6", "#b9b7b0", "#eb6834"
plt.rcParams.update({"font.family": "sans-serif", "font.size": 10, "axes.edgecolor": REJILLA,
                     "axes.labelcolor": TINTA_SUAVE, "xtick.color": TINTA_SUAVE, "ytick.color": TINTA_SUAVE})

def resumen(df, por):
    t = df.groupby(por)[["ingresos", "materiales", "subcontratas", "mano_obra", "margen"]].sum()
    t["margen_pct"] = t.margen / t.ingresos * 100
    t["obras"] = df.groupby(por).size()
    return t

# Rentabilidad por tipo de trabajo
resumen(margen, "tipo").sort_values("margen_pct", ascending=False)

In [ ]:
# Margen (%) por tipo de trabajo, antes y después de la subida de materiales
resumen(margen, ["tipo", "periodo"]).margen_pct.unstack()

In [ ]:
# Peso del material sobre lo facturado (%), antes y después: aquí se ve la subida
(margen.materiales / margen.ingresos * 100).groupby([margen.tipo, margen.periodo]).mean().unstack()

In [ ]:
# Gráfico 1: margen por tipo de trabajo, antes y después de la subida de materiales
t = resumen(margen, ["tipo", "periodo"]).margen_pct.unstack()
t = t.loc[resumen(margen, "tipo").margen_pct.sort_values().index]
fig, ax = plt.subplots(figsize=(8, 4.2))
y = range(len(t))
antes, despues = t.columns
ax.barh([i + 0.2 for i in y], t[antes], height=0.38, color=GRIS, label=antes)
ax.barh([i - 0.2 for i in y], t[despues], height=0.38, color=AZUL, label=despues)
for i, (a, d) in enumerate(zip(t[antes], t[despues])):
    ax.text(a + 0.6, i + 0.2, f"{a:.0f}%", va="center", fontsize=9, color=TINTA_SUAVE)
    ax.text(d + 0.6, i - 0.2, f"{d:.0f}%", va="center", fontsize=9, color=TINTA, fontweight="bold")
ax.set_yticks(list(y), t.index, color=TINTA)
ax.set_xlabel("Margen de contribución sobre ingresos (%)")
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
fig.suptitle("Las cocinas han perdido 10 puntos de margen\ndesde la subida de materiales",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_margen_por_tipo.png", dpi=200)
plt.show()

### Margen por cliente
Se separa a Habitat Mar Inversiones, el cliente más grande, del resto de clientes de su tipo.

In [ ]:
# Gráfico 2: margen por tipo de cliente, con Habitat Mar aparte
margen["grupo"] = margen.cliente.where(margen.cliente.str.startswith("Habitat"),
                                margen.tipo_cliente.map({"Particular": "Particulares", "Empresa": "Otras empresas", "Comunidad": "Comunidades"}))
t2 = resumen(margen, "grupo").sort_values("margen_pct")
fig, ax = plt.subplots(figsize=(8, 3.1))
colores = [AZUL if g.startswith("Habitat") else GRIS for g in t2.index]
ax.barh(t2.index, t2.margen_pct, color=colores, height=0.6)
for i, (pct, ing) in enumerate(zip(t2.margen_pct, t2.ingresos)):
    ax.text(pct + 0.6, i, f"{pct:.0f}%  ·  {ing/1000:,.0f} mil € facturados".replace(",", "."),
            va="center", fontsize=9, color=TINTA_SUAVE)
ax.set_yticks(range(len(t2)), t2.index, color=TINTA)
ax.set_xlim(0, t2.margen_pct.max() * 1.6)
ax.set_xlabel("Margen de contribución sobre ingresos (%)")
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
fig.suptitle("Habitat Mar deja menos de la mitad de margen que el resto de clientes",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_margen_por_cliente.png", dpi=200)
plt.show()

In [ ]:
# Margen por hora de trabajo de los operarios: cuánto deja cada hora que se dedica a cada tipo de obra
# (las horas de los operarios son limitadas: es la medida que importa si hay que elegir qué obras aceptar)
t = margen.assign(es_habitat=margen.cliente.str.startswith("Habitat"))
pd.concat([
    t.groupby("tipo")[["margen", "horas"]].sum(),
    t[t.tipo == "Reforma integral"].groupby("es_habitat")[["margen", "horas"]].sum()
     .rename(index={True: "Reforma integral · Habitat Mar", False: "Reforma integral · resto de clientes"}),
]).assign(margen_por_hora=lambda d: d.margen / d.horas).sort_values("margen_por_hora", ascending=False)

## Mis observaciones · Rentabilidad

**Margen total**
- Las 220 obras terminadas facturaron 1.644.925 € y dejaron 434.212 € de margen de contribución (26%).
  De ahí salen los gastos de estructura y el beneficio.

**La subida de materiales**
- **Cocinas: del 30% al 20% de margen.** El material ha pasado del 39% al 54% de lo facturado,
  y Jordi apenas ha subido precios.
- **Baños: del 37% al 34%.** También más material (del 26% al 35%), pero lo aguantan mejor.
- Pintura, mantenimiento y locales no se ven afectados: usan poco material.

**Habitat Mar Inversiones**
- Es el cliente que más factura (339.099 €), pero deja solo un **12% de margen**: menos de la mitad
  que el resto de clientes (29% particulares, 36% otras empresas).
- Todas sus obras son reformas integrales con un 8% de descuento. Incluso comparando solo reformas
  integrales, deja menos que un particular (12% frente a 17%).
- **Cada hora de un operario en una obra de Habitat deja 13 €; en el resto de obras, 30 €.**
  Habitat ocupa 3.235 horas, el 20% de todas las horas de los operarios.

**Qué trabajos interesan más**
- Por margen sobre lo facturado: pintura (49%), mantenimiento (39%) y baños (36%).
- Por margen por hora: locales (36 €/h), baños y cocinas (34 €/h). Mantenimiento deja poco por hora
  (21 €/h) aunque su porcentaje parezca alto, porque son trabajos pequeños.
- Las reformas integrales son las que menos dejan: 14% de margen y 15 €/h.

**Pendiente**
- Este margen no incluye los gastos de estructura. Se verán en la parte de caja.

## 2. Cobros: quién debe cuánto y quién paga tarde

Se usa la conciliación bancaria de `04_limpieza`: cada cobro del banco está unido a su factura,
así que se sabe exactamente cuántos días pasaron entre la factura y el cobro.

- **Días hasta el cobro:** desde la fecha de la factura hasta el día en que entró el dinero.
- **Días de retraso:** desde el vencimiento pactado hasta el cobro (negativo = pagó antes de tiempo).
- Las medias están **ponderadas por importe**: una factura de 20.000 € pesa más que una de 300 €.
- Los particulares pagan en el momento (su vencimiento es el día de la factura); las empresas
  tienen 30 o 60 días según lo pactado.

In [ ]:
# Cada trozo de cobro con su factura: cuántos días pasaron desde la factura hasta el cobro
pagos = sql("""
SELECT f.num_factura, f.fecha AS fecha_factura, f.vencimiento, f.cliente_id,
       cl.cliente, cl.tipo_cliente, c.fecha AS fecha_cobro, cf.importe,
       JULIANDAY(c.fecha) - JULIANDAY(f.fecha) AS dias_hasta_cobro,
       JULIANDAY(c.fecha) - JULIANDAY(f.vencimiento) AS dias_de_retraso
FROM limpio_cobros_facturas AS cf
JOIN limpio_cobros AS c USING (fila_banco)
JOIN limpio_facturas AS f ON f.num_factura = cf.num_factura
JOIN limpio_clientes AS cl ON cl.cliente_id = f.cliente_id
""")
pagos["grupo"] = pagos.cliente.where(pagos.tipo_cliente != "Particular", "Particulares")

def media_ponderada(df, columna):
    # Media ponderada por importe: una factura de 20.000 € pesa más que una de 300 €
    return (df[columna] * df.importe).sum() / df.importe.sum()

In [ ]:
dias = (pagos.groupby("grupo")
        .apply(lambda d: pd.Series({
            "cobrado": d.importe.sum(),
            "dias_hasta_cobro": media_ponderada(d, "dias_hasta_cobro"),
            "plazo_pactado": media_ponderada(d, "dias_hasta_cobro") - media_ponderada(d, "dias_de_retraso"),
            "dias_de_retraso": media_ponderada(d, "dias_de_retraso"),
        }), include_groups=False)
        .sort_values("dias_hasta_cobro"))

fig, ax = plt.subplots(figsize=(8, 4.4))
colores = [AZUL if g.startswith("Habitat") else GRIS for g in dias.index]
ax.barh(dias.index, dias.dias_hasta_cobro, color=colores, height=0.6)
ax.scatter(dias.plazo_pactado, range(len(dias)), marker="|", s=260, linewidths=2.2, color=TINTA, zorder=3,
           label="Plazo de pago pactado")
for i, (d, p, r) in enumerate(zip(dias.dias_hasta_cobro, dias.plazo_pactado, dias.dias_de_retraso)):
    texto = f"{d:.0f} días" + (f"  ·  {r:.0f} de retraso" if r >= 5 else "")
    ax.text(max(d, p) + 3, i, texto, va="center", fontsize=9,
            color=TINTA if dias.index[i].startswith("Habitat") else TINTA_SUAVE)
ax.set_yticks(range(len(dias)), dias.index, color=TINTA)
ax.set_xlim(0, dias.dias_hasta_cobro.max() * 1.45)
ax.set_xlabel("Días medios desde la factura hasta el cobro (ponderados por importe)")
ax.xaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.legend(loc="lower right", frameon=False, fontsize=9)
fig.suptitle("Habitat Mar tarda 139 días en pagar: 79 más de lo pactado",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_dias_de_cobro.png", dpi=200)
plt.show()
dias

### ¿Habitat Mar paga cada vez más tarde?
Ojo al leer esta tabla: solo incluye las facturas **ya cobradas**. Las facturas de Habitat de 2026 que
aún no ha pagado no cuentan, así que la cifra de 2026 parece mejor de lo que es. Es un **sesgo de
supervivencia**: solo se ven las facturas que han llegado a cobrarse.

In [ ]:
p = pagos.assign(
    semestre=pagos.fecha_factura.str[:4] + " · "
             + pagos.fecha_factura.str[5:7].astype(int).map(lambda m: "1er semestre" if m <= 6 else "2º semestre"),
    cliente_grupo=pagos.cliente.where(pagos.cliente.str.startswith("Habitat"), "Resto de clientes"),
)
p.groupby(["semestre", "cliente_grupo"]).apply(
    lambda d: media_ponderada(d, "dias_hasta_cobro"), include_groups=False).unstack()

In [ ]:
# ¿Cuánto dinero le "presta" Jordi a Habitat Mar en cada momento?
# Cobro medio diario de Habitat × días que tarda en pagar = lo que Habitat le debe de media
h = pagos[pagos.cliente.str.startswith("Habitat")]
dias_periodo = (pd.Timestamp("2026-09-30") - pd.Timestamp("2024-01-01")).days
cobro_diario = h.importe.sum() / dias_periodo
pd.DataFrame({
    "cobrado_de_habitat_al_año": [cobro_diario * 365],
    "dinero_que_debe_de_media": [cobro_diario * media_ponderada(h, "dias_hasta_cobro")],
    "de_ello_por_pagar_tarde": [cobro_diario * media_ponderada(h, "dias_de_retraso")],
})

In [ ]:
# Lo que queda por cobrar a 30/09/2026, según cuánto tiempo lleva vencido
sql("""
SELECT CASE
         WHEN JULIANDAY('2026-09-30') - JULIANDAY(vencimiento) <= 0 THEN '1 · Aún no ha vencido'
         WHEN JULIANDAY('2026-09-30') - JULIANDAY(vencimiento) <= 30 THEN '2 · Vencido hace 1-30 días'
         WHEN JULIANDAY('2026-09-30') - JULIANDAY(vencimiento) <= 90 THEN '3 · Vencido hace 31-90 días'
         ELSE '4 · Vencido hace más de 90 días'
       END AS antiguedad,
       COUNT(*) AS facturas,
       ROUND(SUM(total_a_cobrar - cobrado), 2) AS pendiente
FROM estado_facturas
WHERE total_a_cobrar - cobrado > 0.02
GROUP BY antiguedad
""")

## Mis observaciones · Cobros

**Quién paga tarde**
- Los particulares pagan casi en el momento (7 días de media) y la mayoría de empresas cumple su plazo.
- **Habitat Mar Inversiones tarda 139 días en pagar, 79 más de los 60 pactados.** Y va a peor:
  152 días de media en las facturas del segundo semestre de 2025, y 7 facturas de 2026 aún sin pagar.
- La comunidad de propietarios tarda 66 días porque paga en tres plazos (derramas).
- Restaurant Can Mas pagaba con un mes de retraso y ha dejado de pagar.

**Lo que esto le cuesta a Jordi**
- **Habitat le debe, de media, unos 47.000 € en cada momento**, de los que unos 27.000 € se deben
  solo a que paga tarde. Es casi exactamente lo que le falta a Jordi en la cuenta en los peores
  momentos (−45.197 € en mayo de 2026).

**Pendiente de cobro a 30/09/2026: 114.947 €**
- 22.818 € aún no han vencido: es normal.
- 92.129 € están vencidos, y 20.866 € llevan más de 90 días: son las 3 facturas que parecen impagadas.

## 3. Caja e IVA: ¿por qué hay beneficio pero no hay dinero?

El **beneficio** se calcula con lo que se factura y lo que se gasta, aunque aún no se haya cobrado
ni pagado. El **dinero en la cuenta** solo cambia cuando el dinero entra o sale de verdad. La
diferencia entre los dos está en tres cosas: lo que los clientes aún deben, lo que Jordi aún no
ha pagado y los impuestos.

### Cuenta de resultados por año

In [ ]:
# Cuenta de resultados simplificada por año (sin IVA)
anual = lambda consulta: sql(consulta).set_index("anio").iloc[:, 0]
cuenta = pd.DataFrame({
    "facturado": anual("SELECT SUBSTR(fecha, 1, 4) AS anio, SUM(base) FROM limpio_facturas GROUP BY anio"),
    "materiales_y_subcontratas": -anual("""SELECT SUBSTR(fecha, 1, 4) AS anio, SUM(base) FROM limpio_gastos
                                           WHERE categoria IN ('Materiales', 'Subcontratas') GROUP BY anio"""),
    "operarios": -anual("""SELECT SUBSTR(mes, 1, 4) AS anio, SUM(coste_empresa) FROM limpio_nominas
                           WHERE categoria LIKE 'Oficial%' GROUP BY anio"""),
    "gerente_y_administrativa": -anual("""SELECT SUBSTR(mes, 1, 4) AS anio, SUM(coste_empresa) FROM limpio_nominas
                                          WHERE categoria NOT LIKE 'Oficial%' GROUP BY anio"""),
    "otros_gastos_de_estructura": -anual("""SELECT SUBSTR(fecha, 1, 4) AS anio, SUM(base) FROM limpio_gastos
                                            WHERE categoria NOT IN ('Materiales', 'Subcontratas') GROUP BY anio"""),
    "comisiones_e_intereses": anual("""SELECT SUBSTR(fecha, 1, 4) AS anio, SUM(importe) FROM banco
                                       WHERE concepto LIKE 'COMISION%' OR concepto LIKE 'LIQUIDACION%' GROUP BY anio"""),
})
cuenta["beneficio_antes_de_impuestos"] = cuenta.sum(axis=1)
cuenta["beneficio_sobre_facturado_pct"] = cuenta.beneficio_antes_de_impuestos / cuenta.facturado * 100
cuenta.T

### Del beneficio al dinero en la cuenta
Un puente (o gráfico en cascada) explica, paso a paso, cómo se pasa del beneficio a la variación
del saldo del banco. La suma de los pasos tiene que dar exactamente la variación real del saldo:
si no cuadra, falta algo.

In [ ]:
# --- Puente: del beneficio a la variación del dinero en la cuenta (ene 2024 – sep 2026) ---
f = sql("SELECT SUM(base) AS ingresos, SUM(cuota_iva) AS iva, SUM(total) AS total FROM limpio_facturas").iloc[0]
g = sql("""SELECT SUM(base) AS base, SUM(cuota_iva) AS iva, SUM(retencion_irpf) AS retenciones,
                  SUM(total_a_pagar) AS total FROM limpio_gastos""").iloc[0]
n = sql("""SELECT SUM(coste_empresa) AS coste, SUM(ss_empresa + ss_trabajador) AS ss,
                  SUM(irpf) AS irpf, SUM(liquido) AS neto FROM limpio_nominas""").iloc[0]
b = sql("""
SELECT CASE
         WHEN concepto LIKE 'TRANSF DE %' OR concepto LIKE 'BIZUM DE %' THEN 'cobros'
         WHEN concepto LIKE 'TRANSF NOMINAS%' THEN 'nominas'
         WHEN concepto LIKE 'TGSS%' THEN 'seguridad_social'
         WHEN concepto LIKE 'AEAT MOD 303%' THEN 'iva'
         WHEN concepto LIKE 'AEAT MOD 111%' OR concepto LIKE 'AEAT MOD 115%' THEN 'retenciones'
         WHEN concepto LIKE 'AEAT MOD 20%' THEN 'impuesto_sociedades'
         WHEN concepto LIKE 'TRANSF A %' OR concepto LIKE 'RECIBO %' OR concepto LIKE 'PAGO TARJETA %' THEN 'proveedores'
         ELSE 'banco'
       END AS tipo,
       SUM(importe) AS importe
FROM banco GROUP BY tipo
""").set_index("tipo").importe

beneficio = f.ingresos - g.base - n.coste + b.banco             # sin IVA, antes del impuesto de sociedades
pendiente_cobro = f.total - b.cobros                             # facturado con IVA que aún no ha entrado
aun_sin_pagar = ((g.total + b.proveedores)                       # facturas de proveedores aún sin pagar
                 + (n.neto + b.nominas)                          # nóminas aún sin pagar
                 + (n.ss + b.seguridad_social)                   # Seguridad Social del último mes
                 + (f.iva - g.iva + b.iva)                       # IVA del último trimestre, aún sin ingresar
                 + (g.retenciones + n.irpf + b.retenciones))     # retenciones de IRPF aún sin ingresar
puente = pd.Series({
    "Beneficio antes de impuestos": beneficio,
    "Impuesto de sociedades pagado": b.impuesto_sociedades,
    "Facturas que los clientes aún deben": -pendiente_cobro,
    "Pagos que Jordi aún no ha hecho": aun_sin_pagar,
})
variacion_caja = b.sum()
# Comprobación: el puente tiene que llegar exactamente a la variación real del saldo
display(pd.DataFrame({"importe": puente, "acumulado": puente.cumsum()}))
pd.DataFrame({"suma_del_puente": [puente.sum()], "variacion_real_de_la_cuenta": [variacion_caja]})

In [ ]:
# Gráfico en cascada del puente
import textwrap
euros = lambda v: f"{v:,.0f}".replace(",", ".") + " €"
pasos = list(puente.items()) + [("Variación del dinero en la cuenta", variacion_caja)]
fig, ax = plt.subplots(figsize=(9, 4.8))
acumulado = 0
for i, (nombre, valor) in enumerate(pasos):
    total = i in (0, len(pasos) - 1)
    base = 0 if total else acumulado
    color = TINTA_SUAVE if total else (AZUL if valor > 0 else NARANJA)
    ax.bar(i, valor, bottom=base, color=color, width=0.6)
    tope = base + valor if valor > 0 else base
    ax.text(i, max(tope, 0) + 2500, ("+" if valor > 0 else "−") + euros(abs(valor)), ha="center", fontsize=9,
            color=TINTA, fontweight="bold")
    if not total or i == 0:
        acumulado = acumulado + valor if i else valor
    if i < len(pasos) - 1:
        ax.plot([i + 0.3, i + 0.7], [acumulado, acumulado], color=TINTA_SUAVE, linewidth=0.8, linestyle=":")
ax.axhline(0, color=TINTA_SUAVE, linewidth=0.8)
ax.set_xticks(range(len(pasos)), [textwrap.fill(p[0], 18) for p in pasos], fontsize=9, color=TINTA)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:,.0f} mil €".replace(",", ".")))
ax.yaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
fig.suptitle(f"De enero de 2024 a septiembre de 2026 ganó {euros(beneficio)},\npero tiene {euros(-variacion_caja)} menos en la cuenta",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
ax.set_ylim(bottom=ax.get_ylim()[0] * 1.06, top=max(beneficio, 0) * 1.18)
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_beneficio_vs_caja.png", dpi=200)
plt.show()

### ¿Cuándo se tensa la caja?

In [ ]:
# Saldo de la cuenta al final de cada día, y lo que Habitat Mar debía ese mismo día
saldo = (sql("SELECT fila, fecha, saldo FROM banco ORDER BY fila DESC")
         .groupby("fecha").saldo.last())
saldo.index = pd.to_datetime(saldo.index)
dias = pd.date_range("2024-01-01", "2026-09-30")
saldo = saldo.reindex(dias).ffill().fillna(38000)

facturado_h = sql("""SELECT fecha, SUM(total) AS importe FROM limpio_facturas
                     WHERE cliente_id = (SELECT cliente_id FROM limpio_clientes WHERE cliente LIKE 'Habitat%')
                     GROUP BY fecha""")
cobrado_h = sql("""SELECT c.fecha, SUM(c.importe) AS importe FROM limpio_cobros AS c
                   WHERE c.cliente_id = (SELECT cliente_id FROM limpio_clientes WHERE cliente LIKE 'Habitat%')
                   GROUP BY c.fecha""")
serie = lambda df: df.assign(fecha=pd.to_datetime(df.fecha)).set_index("fecha").importe.reindex(dias, fill_value=0).cumsum()
deuda_habitat = serie(facturado_h) - serie(cobrado_h)

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axhline(0, color=TINTA_SUAVE, linewidth=0.8)
ax.fill_between(dias, saldo, 0, where=saldo < 0, color=AZUL, alpha=0.15, linewidth=0)
ax.plot(dias, saldo, color=AZUL, linewidth=2, label="Saldo de la cuenta")
ax.plot(dias, deuda_habitat, color=NARANJA, linewidth=2, label="Lo que debe Habitat Mar")
minimo = saldo.idxmin()
ax.annotate(f"{saldo.min():,.0f} €".replace(",", "."), (minimo, saldo.min()), xytext=(10, -4),
            textcoords="offset points", fontsize=9, color=TINTA, fontweight="bold")
ax.text(dias[-1], saldo.iloc[-1], "  Saldo", color=AZUL, va="center", fontsize=9, fontweight="bold")
ax.text(dias[-1], deuda_habitat.iloc[-1], "  Debe Habitat", color=NARANJA, va="center", fontsize=9, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:,.0f} mil €".replace(",", ".")))
ax.yaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.set_xlim(dias[0], dias[-1] + pd.Timedelta(days=110))
ax.legend(loc="upper left", frameon=False, fontsize=9, ncol=2)
fig.suptitle("Cuanto más le debe Habitat Mar, más se hunde la cuenta",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_saldo_y_deuda_habitat.png", dpi=200)
plt.show()

# ¿Van juntas las dos líneas? Correlación entre el saldo y la deuda de Habitat
pd.DataFrame({"correlacion_saldo_vs_deuda_habitat": [saldo.corr(deuda_habitat)]})

### El IVA: Jordi paga a Hacienda un IVA que aún no ha cobrado
El IVA se paga a Hacienda por las facturas **emitidas** en el trimestre, se hayan cobrado o no.
Si un cliente paga tarde, Jordi adelanta de su bolsillo el IVA de esa factura.

In [ ]:
# IVA de cada trimestre: lo que se cobra a los clientes menos lo que se paga a proveedores.
# Y la pregunta clave: cuando Jordi paga el IVA a Hacienda, ¿cuánto de ese IVA aún no lo ha cobrado?
trimestre = lambda col: f"SUBSTR({col}, 1, 4) || '-T' || ((CAST(SUBSTR({col}, 6, 2) AS INTEGER) - 1) / 3 + 1)"
iva = (sql(f"SELECT {trimestre('fecha')} AS trimestre, SUM(cuota_iva) AS iva_facturado FROM limpio_facturas GROUP BY trimestre")
       .merge(sql(f"SELECT {trimestre('fecha')} AS trimestre, SUM(cuota_iva) AS iva_de_gastos FROM limpio_gastos GROUP BY trimestre"),
              on="trimestre"))
pagos_303 = sql("SELECT fecha AS fecha_pago, concepto, -importe AS iva_pagado FROM banco WHERE concepto LIKE 'AEAT MOD 303%'")
pagos_303["trimestre"] = pagos_303.concepto.str.extract(r"(\d)T (\d{4})").apply(lambda r: f"{r[1]}-T{r[0]}", axis=1)
iva = iva.merge(pagos_303[["trimestre", "fecha_pago", "iva_pagado"]], on="trimestre", how="left")

facturas_t = sql(f"SELECT num_factura, total, cuota_iva, {trimestre('fecha')} AS trimestre FROM limpio_facturas WHERE rectifica_a IS NULL")
cobros_f = sql("SELECT cf.num_factura, c.fecha, cf.importe FROM limpio_cobros_facturas AS cf JOIN limpio_cobros AS c USING (fila_banco)")
def iva_sin_cobrar(fila):
    if pd.isna(fila.fecha_pago):
        return None
    total = 0
    for f in facturas_t[facturas_t.trimestre == fila.trimestre].itertuples():
        cobrado = cobros_f[(cobros_f.num_factura == f.num_factura) & (cobros_f.fecha <= fila.fecha_pago)].importe.sum()
        total += f.cuota_iva * max(0, 1 - cobrado / f.total)
    return total
iva["iva_aun_sin_cobrar_al_pagar"] = iva.apply(iva_sin_cobrar, axis=1)
iva

## Mis observaciones · Caja e IVA

**Beneficio**
- La empresa gana dinero, pero poco: entre 16.000 y 26.000 € al año antes de impuestos,
  alrededor de un 3-5% de lo que factura. El margen de las obras (26%) se lo comen el sueldo
  del gerente y la administrativa y los gastos de estructura (alquiler, furgonetas, gasoil...).

**Por qué no hay dinero**
- De enero de 2024 a septiembre de 2026 ganó 63.320 €, pero tiene 3.509 € menos en la cuenta.
- 14.590 € se fueron en el impuesto de sociedades.
- **114.947 € son facturas que los clientes aún deben**: es la causa principal.
- Lo compensa en parte que Jordi también debe 62.708 € (proveedores que cobran a 30 días,
  la Seguridad Social del último mes y el IVA y las retenciones del último trimestre, que se pagan
  en octubre). Ese dinero está en la cuenta, pero no es suyo.

**Cuándo se tensa la caja**
- **El saldo y lo que debe Habitat Mar se mueven como un espejo**: cuando la deuda de Habitat sube,
  la cuenta baja. En la primavera de 2026 Habitat llegó a deber más de 140.000 € y la cuenta cayó a
  −45.197 € (mayo de 2026).

**IVA**
- En varios trimestres, cuando Jordi pagó el IVA a Hacienda, buena parte de ese IVA aún no lo había
  cobrado: por ejemplo, en abril de 2026 pagó 10.883 € de IVA mientras los clientes aún le debían
  16.929 € de IVA de ese trimestre. Es dinero que adelanta él.

## 4. Previsión de caja: octubre a diciembre de 2026

Se apunta cada entrada y salida de dinero previsible con su fecha y se va sumando al saldo del
30 de septiembre. Es la técnica estrella del caso: responde a la pregunta que más preocupa a Jordi,
**¿voy a poder pagar?**

**Supuestos** (todos salen de los datos de la empresa; cambiar uno es cambiar una línea del código)
- **Facturas pendientes:** cada una se cobra cuando ese cliente suele pagar (fecha de la factura +
  sus días medios de cobro). Las 3 facturas vencidas hace más de 90 días se dan por impagadas.
- **Facturación nueva:** la del mismo mes de 2024 y 2025 (la actividad tiene temporadas), ajustada
  por cómo va 2026 frente a 2025. Se cobra a los días medios de los clientes que no son Habitat.
  Lo que facture Habitat ahora no se cobrará antes de enero, porque tarda 139 días en pagar.
- **Compras de material y subcontratas:** el mismo porcentaje sobre lo facturado que en 2026,
  con IVA, pagadas el mes siguiente.
- **Nóminas y Seguridad Social:** las de septiembre, con la paga extra en diciembre.
- **Impuestos del 3er trimestre** (20 de octubre): el IVA y las retenciones reales del trimestre,
  y los pagos a cuenta del impuesto de sociedades de octubre y diciembre.
- **Gastos fijos:** la media de julio a septiembre de 2026 (alquiler, renting, gasoil, gestoría...).

**Escenarios**
- **Base:** todo el mundo paga como suele.
- **Optimista:** Habitat paga en noviembre todo lo que tiene vencido.
- **Pesimista:** Habitat no paga nada y se factura un 15% menos.

In [ ]:
# ============ PREVISIÓN DE CAJA · OCTUBRE A DICIEMBRE DE 2026 ============
# Se apunta cada entrada y salida de dinero previsible con su fecha, y se va sumando al saldo.
INICIO, FIN = pd.Timestamp("2026-10-01"), pd.Timestamp("2026-12-31")
saldo_inicial = sql("SELECT saldo FROM banco ORDER BY fila LIMIT 1").iloc[0, 0]
id_habitat = sql("SELECT cliente_id FROM limpio_clientes WHERE cliente LIKE 'Habitat%'").iloc[0, 0]

def ultimo_laborable(anio, mes):
    d = pd.Timestamp(anio, mes, 1) + pd.offsets.MonthEnd(0)
    while d.weekday() >= 5:
        d -= pd.Timedelta(days=1)
    return d

movimientos = []                     # (fecha, concepto, tipo, importe, escenario)
def apunta(fecha, concepto, tipo, importe, escenario="todos"):
    movimientos.append((pd.Timestamp(fecha), concepto, tipo, importe, escenario))

# --- 1. Cobro de las facturas pendientes -------------------------------------------------
# Cada cliente cobra cuando suele hacerlo: fecha de la factura + sus días medios de cobro.
dias_cliente = pagos.groupby("cliente_id").apply(lambda d: media_ponderada(d, "dias_hasta_cobro"), include_groups=False)
dias_tipo = pagos.groupby("tipo_cliente").apply(lambda d: media_ponderada(d, "dias_hasta_cobro"), include_groups=False)
pendientes = sql("""SELECT e.num_factura, e.fecha, e.vencimiento, e.cliente_id, c.tipo_cliente,
                           e.total_a_cobrar - e.cobrado AS pendiente
                    FROM estado_facturas AS e JOIN limpio_clientes AS c USING (cliente_id)
                    WHERE e.total_a_cobrar - e.cobrado > 0.02""")
for f in pendientes.itertuples():
    vencida_hace = (pd.Timestamp("2026-09-30") - pd.Timestamp(f.vencimiento)).days
    if vencida_hace > 90:
        continue                     # más de 90 días vencida y sin ningún pago: se da por impagada
    dias = dias_cliente.get(f.cliente_id, dias_tipo[f.tipo_cliente])
    fecha = max(pd.Timestamp(f.fecha) + pd.Timedelta(days=round(dias)), INICIO)
    if f.cliente_id == id_habitat:
        apunta(fecha, f"Cobro {f.num_factura}", "Cobros de facturas pendientes", f.pendiente, "base")
        apunta(fecha, f"Cobro {f.num_factura}", "Cobros de facturas pendientes", 0, "pesimista")
        apunta(min(fecha, pd.Timestamp("2026-11-16")), f"Cobro {f.num_factura}", "Cobros de facturas pendientes",
               f.pendiente, "optimista")
    else:
        apunta(fecha, f"Cobro {f.num_factura}", "Cobros de facturas pendientes", f.pendiente)

# --- 2. Facturación nueva de octubre a diciembre y su cobro ----------------------------
# Se estima con lo facturado en el mismo mes de 2024 y 2025 (la actividad tiene temporadas),
# ajustado por cómo va 2026 frente a 2025. Habitat se trata aparte: tarda 139 días en pagar,
# así que lo que facture ahora no se cobra antes de diciembre.
fact = sql("SELECT fecha, cliente_id, base, total FROM limpio_facturas")
fact["fecha"] = pd.to_datetime(fact.fecha)
fact["es_habitat"] = fact.cliente_id == id_habitat
resto = fact[~fact.es_habitat]
crecimiento = (resto[(resto.fecha >= "2026-01-01")].total.sum()
               / resto[(resto.fecha >= "2025-01-01") & (resto.fecha <= "2025-09-30")].total.sum())
dias_resto = media_ponderada(pagos[pagos.cliente_id != id_habitat], "dias_hasta_cobro")
ratio_compras = (sql("SELECT SUM(base) FROM limpio_gastos WHERE categoria IN ('Materiales', 'Subcontratas') AND fecha >= '2026-01-01'").iloc[0, 0]
                 / fact[fact.fecha >= "2026-01-01"].base.sum())
habitat_mensual = fact[fact.es_habitat & (fact.fecha >= "2026-01-01")].base.sum() / 9
base_sobre_total = resto.base.sum() / resto.total.sum()          # para quitar el IVA (mezcla de 10% y 21%)
facturacion_prevista = {}
for mes in (10, 11, 12):
    mismo_mes = resto[resto.fecha.dt.month == mes].groupby(resto.fecha.dt.year).total.sum()
    previsto = mismo_mes.mean() * crecimiento
    facturacion_prevista[mes] = previsto
    # Las facturas se reparten a lo largo del mes (una cuarta parte cada semana)
    # y cada una se cobra a los días medios de cobro de los clientes que no son Habitat
    for dia in (4, 11, 18, 25):
        fecha = pd.Timestamp(2026, mes, dia) + pd.Timedelta(days=round(dias_resto))
        concepto = f"Cobro facturación nueva {mes}/2026"
        apunta(fecha, concepto, "Cobros de facturación nueva", previsto / 4, "base")
        apunta(fecha, concepto, "Cobros de facturación nueva", previsto / 4, "optimista")
        apunta(fecha, concepto, "Cobros de facturación nueva", previsto / 4 * 0.85, "pesimista")
    # Compras de material y subcontratas para esas obras (también las de Habitat), con IVA.
    # Se pagan el mes siguiente, repartidas entre los días 5 y 25 (cada proveedor cobra un día)
    compras = (previsto * base_sobre_total + habitat_mensual) * ratio_compras * 1.21
    if mes < 12:
        for dia in (5, 10, 15, 20, 25):
            apunta(pd.Timestamp(2026, mes + 1, dia), f"Proveedores de {mes}/2026", "Proveedores y subcontratas", -compras / 5)

# --- 3. Lo que Jordi ya debía el 30 de septiembre: proveedores (se pagan en octubre) ----
pendiente_proveedores = (sql("SELECT SUM(total_a_pagar) FROM limpio_gastos").iloc[0, 0]
                         + sql("""SELECT SUM(importe) FROM banco WHERE concepto LIKE 'TRANSF A %'
                                  OR concepto LIKE 'RECIBO %' OR concepto LIKE 'PAGO TARJETA %'""").iloc[0, 0])
for dia in (5, 10, 15, 20, 25):
    apunta(pd.Timestamp(2026, 10, dia), "Proveedores de septiembre", "Proveedores y subcontratas", -pendiente_proveedores / 5)

# --- 4. Nóminas y Seguridad Social (diciembre lleva paga extra) --------------------------
ultima = sql("SELECT SUM(liquido) AS neto, SUM(ss_empresa + ss_trabajador) AS ss FROM limpio_nominas WHERE mes = '2026-09'").iloc[0]
for mes in (10, 11, 12):
    apunta(ultimo_laborable(2026, mes), f"Nóminas {mes}/2026", "Nóminas y Seguridad Social",
           -ultima.neto * (2 if mes == 12 else 1))
    apunta(ultimo_laborable(2026, mes), f"Seguridad Social {mes - 1}/2026", "Nóminas y Seguridad Social", -ultima.ss)

# --- 5. Impuestos del tercer trimestre (20 de octubre) y pagos a cuenta de sociedades ---
t3 = lambda tabla, col: sql(f"SELECT SUM({col}) FROM {tabla} WHERE fecha BETWEEN '2026-07-01' AND '2026-09-30'").iloc[0, 0]
iva_t3 = t3("limpio_facturas", "cuota_iva") - t3("limpio_gastos", "cuota_iva")
retenciones_t3 = (t3("limpio_gastos", "retencion_irpf")
                  + sql("SELECT SUM(irpf) FROM limpio_nominas WHERE mes BETWEEN '2026-07' AND '2026-09'").iloc[0, 0])
pago_202 = -sql("SELECT importe FROM banco WHERE concepto LIKE 'AEAT MOD 202 2026%' ORDER BY fila LIMIT 1").iloc[0, 0]
apunta("2026-10-20", "IVA 3T (modelo 303)", "Impuestos", -max(iva_t3, 0))
apunta("2026-10-20", "Retenciones 3T (modelos 111 y 115)", "Impuestos", -retenciones_t3)
apunta("2026-10-20", "Pago a cuenta sociedades (202)", "Impuestos", -pago_202)
apunta("2026-12-21", "Pago a cuenta sociedades (202)", "Impuestos", -pago_202)

# --- 6. Gastos fijos de cada mes: la media de lo que se pagó de julio a septiembre de 2026
fijos = sql("""SELECT CASE WHEN concepto LIKE 'COMISION%' THEN 'COMISIONES' ELSE concepto END AS concepto,
                      SUM(importe) / 3 AS importe_mensual
               FROM banco
               WHERE fecha BETWEEN '2026-07-01' AND '2026-09-30'
                 AND (concepto LIKE 'RECIBO %' OR concepto LIKE 'PAGO TARJETA %' OR concepto LIKE 'COMISION%')
               GROUP BY 1""")
for mes in (10, 11, 12):
    for g in fijos.itertuples():
        apunta(pd.Timestamp(2026, mes, 10), g.concepto, "Gastos fijos", g.importe_mensual)

# --- 7. Saldo día a día en cada escenario ------------------------------------------------
mov = pd.DataFrame(movimientos, columns=["fecha", "concepto", "tipo", "importe", "escenario"])
dias = pd.date_range(INICIO, FIN)
escenarios = {}
for e in ("pesimista", "base", "optimista"):
    m = mov[mov.escenario.isin(["todos", e]) & (mov.fecha <= FIN)]
    escenarios[e] = saldo_inicial + m.groupby("fecha").importe.sum().reindex(dias, fill_value=0).cumsum()
prevision = pd.DataFrame(escenarios)

In [ ]:
import matplotlib.dates as mdates
euros = lambda v: f"{v:,.0f}".replace(",", ".") + " €"
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.axhline(0, color=TINTA_SUAVE, linewidth=0.8)
estilos = {"optimista": (GRIS, "--", "Optimista: Habitat paga lo vencido en noviembre"),
           "base": (AZUL, "-", "Base: cada cliente paga como suele"),
           "pesimista": (NARANJA, "--", "Pesimista: Habitat no paga y se factura un 15% menos")}
for e, (color, linea, etiqueta) in estilos.items():
    ax.plot(prevision.index, prevision[e], color=color, linestyle=linea, linewidth=2, label=etiqueta)
    ax.text(prevision.index[-1] + pd.Timedelta(days=2), prevision[e].iloc[-1], euros(prevision[e].iloc[-1]),
            color=color, va="center", fontsize=9, fontweight="bold")
ax.fill_between(prevision.index, prevision.pesimista, 0, where=prevision.pesimista < 0,
                color=NARANJA, alpha=0.12, linewidth=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v/1000:,.0f} mil €".replace(",", ".")))
ax.yaxis.grid(True, color=REJILLA, linewidth=0.8); ax.set_axisbelow(True)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
ax.set_xlim(prevision.index[0], prevision.index[-1] + pd.Timedelta(days=14))
ax.legend(loc="lower left", frameon=False, fontsize=9)
fig.suptitle(f"Si Habitat Mar no paga, la cuenta acabará el año en {euros(prevision.pesimista.iloc[-1])}",
             x=0.02, ha="left", fontsize=12, color=TINTA, fontweight="bold")
ax.set_title("Previsión del saldo de la cuenta, día a día, de octubre a diciembre de 2026",
             loc="left", fontsize=9, color=TINTA_SUAVE)
plt.tight_layout()
plt.savefig(PROYECTO / "informe" / "grafico_prevision_caja.png", dpi=200)
plt.show()

# Resumen de cada escenario
pd.DataFrame({
    "saldo_minimo": prevision.min(),
    "dia_del_minimo": prevision.idxmin().dt.strftime("%d/%m/%Y"),
    "dias_en_negativo": (prevision < 0).sum(),
    "saldo_31_diciembre": prevision.iloc[-1],
}).loc[["pesimista", "base", "optimista"]]

In [ ]:
# Detalle de la previsión base: entradas y salidas de cada mes
base = mov[mov.escenario.isin(["todos", "base"]) & (mov.fecha <= FIN)]
base.pivot_table(index="tipo", columns=base.fecha.dt.strftime("%m/%Y"), values="importe", aggfunc="sum", fill_value=0)

## 5. ¿Se puede permitir contratar a otro operario en 2027?

Para responder hay que mirar tres cosas: si los operarios actuales ya están a tope, cuánto dejaría
un operario más y si la caja lo aguanta.

In [ ]:
# ¿Están los operarios a tope? Horas de los partes frente a la jornada anual del convenio
JORNADA_ANUAL = 1736          # horas al año, Convenio General del Sector de la Construcción
capacidad = sql("""
SELECT SUBSTR(semana_inicio, 1, 4) AS anio,
       COUNT(DISTINCT SUBSTR(semana_inicio, 1, 7)) AS meses_con_datos,
       SUM(horas) / 4 AS horas_por_operario
FROM limpio_partes GROUP BY anio
""")
capacidad["horas_por_operario_al_año"] = capacidad.horas_por_operario * 12 / capacidad.meses_con_datos
capacidad["ocupacion_pct"] = capacidad.horas_por_operario_al_año / JORNADA_ANUAL * 100
capacidad

In [ ]:
# ¿Cuántas horas de operario se lleva Habitat Mar cada año?
sql("""
SELECT SUBSTR(p.semana_inicio, 1, 4) AS anio,
       SUM(CASE WHEN c.cliente LIKE 'Habitat%' THEN p.horas ELSE 0 END) AS horas_habitat,
       SUM(p.horas) AS horas_totales,
       ROUND(100.0 * SUM(CASE WHEN c.cliente LIKE 'Habitat%' THEN p.horas ELSE 0 END) / SUM(p.horas), 1) AS pct_habitat
FROM limpio_partes AS p
JOIN limpio_obras AS o ON o.codigo = p.obra
JOIN limpio_clientes AS c USING (cliente_id)
GROUP BY anio
""")

In [ ]:
# ¿Cuánto deja un operario más? Depende de en qué obras trabaje.
# Margen bruto por hora = (facturado − materiales − subcontratas) / horas: lo que deja cada hora
# ANTES de pagar al operario. A eso se le resta lo que cuesta el operario nuevo.
nuevo = sql("SELECT devengado, ss_empresa FROM limpio_nominas WHERE trabajador = 'Abdel Benali' AND mes = '2026-09'").iloc[0]
coste_operario = (nuevo.devengado * 14) * (1 + nuevo.ss_empresa / nuevo.devengado)     # 14 pagas + Seguridad Social
horas_productivas = capacidad[capacidad.anio == "2025"].horas_por_operario_al_año.iloc[0]

m = margen.assign(cliente_grupo=margen.cliente.where(margen.cliente.str.startswith("Habitat"), "Resto de clientes"))
por_hora = m.groupby("cliente_grupo")[["ingresos", "margen", "mano_obra", "horas"]].sum()
por_hora["margen_bruto_por_hora"] = (por_hora.margen + por_hora.mano_obra) / por_hora.horas
por_hora["facturado_por_hora"] = por_hora.ingresos / por_hora.horas

escenarios_contratar = pd.DataFrame({
    "margen_bruto_por_hora": por_hora.margen_bruto_por_hora,
    "horas_al_año": horas_productivas,
    "coste_del_operario": coste_operario,
})
escenarios_contratar["margen_que_aporta"] = escenarios_contratar.margen_bruto_por_hora * escenarios_contratar.horas_al_año
escenarios_contratar["ganancia_neta_al_año"] = escenarios_contratar.margen_que_aporta - escenarios_contratar.coste_del_operario
escenarios_contratar["facturacion_extra_necesaria"] = por_hora.facturado_por_hora * horas_productivas
escenarios_contratar.rename(index={"Resto de clientes": "Si trabaja en obras de particulares y otras empresas",
                                   "Habitat Mar Inversiones S.L.": "Si trabaja en obras de Habitat Mar"})

### La alternativa: liberar las horas que se lleva Habitat
Antes de contratar, conviene ver cuánto ganaría Jordi si las horas que hoy dedica a Habitat Mar
las dedicara a obras de otros clientes (o si renegociara con Habitat).

In [ ]:
# La alternativa sin contratar: dedicar las horas que hoy se lleva Habitat a obras de otros clientes
horas_habitat_2026 = sql("""
SELECT SUM(p.horas) * 12 / 9 FROM limpio_partes AS p
JOIN limpio_obras AS o ON o.codigo = p.obra JOIN limpio_clientes AS c USING (cliente_id)
WHERE c.cliente LIKE 'Habitat%' AND p.semana_inicio LIKE '2026%'
""").iloc[0, 0]
diferencia_por_hora = (por_hora.loc["Resto de clientes", "margen_bruto_por_hora"]
                       - por_hora.loc["Habitat Mar Inversiones S.L.", "margen_bruto_por_hora"])
pd.DataFrame({
    "horas_al_año_en_obras_de_habitat": [horas_habitat_2026],
    "equivale_a_operarios": [horas_habitat_2026 / horas_productivas],
    "margen_extra_por_hora_en_otras_obras": [diferencia_por_hora],
    "margen_extra_al_año": [horas_habitat_2026 * diferencia_por_hora],
})

## Mis observaciones · Previsión y contratación

**Previsión de caja (octubre-diciembre de 2026)**
- **En el escenario base, Jordi llega a fin de año sin números rojos, pero muy justo:** el saldo
  baja a unos 10.700 € a mediados de noviembre y acaba diciembre en unos 14.500 €, tras pagar la
  paga extra.
- **Todo depende de Habitat Mar.** Si paga lo vencido en noviembre, acaba el año con unos 36.000 €.
  Si no paga y además baja un poco el trabajo, la cuenta entra en negativo a finales de octubre y
  acaba el año en unos −52.000 €.
- La diferencia entre el mejor y el peor escenario a 31 de diciembre (88.000 €) la decide un solo cliente.

**Contratar a otro operario**
- **Los 4 operarios ya están casi a tope:** trabajan unas 1.590 horas al año, el 92% de la jornada del
  convenio (1.736 horas). Sin contratar, no se puede crecer.
- Un oficial de 2ª cuesta unos 37.000 € al año a la empresa.
- **Si trabaja en obras de particulares y otras empresas, dejaría unos 50.000 € al año** después de
  pagar su sueldo. Pero necesitaría unos 160.000 € más de facturación al año para tenerlo ocupado:
  hay que confirmar con Jordi que hay clientes esperando.
- Si trabaja en obras de Habitat, dejaría solo unos 23.000 €, y además empeoraría la caja, porque
  Habitat paga a 139 días.

**La alternativa sin contratar**
- Habitat se lleva unas 1.800 horas al año en 2026: **más de un operario entero**.
- Dedicando esas horas a obras de otros clientes, Jordi ganaría unos 32.000 € más al año sin contratar
  a nadie, y dejaría de financiar a su cliente más moroso.

**Conclusión**
- **Sí puede contratar, pero no todavía.** Antes tiene que cobrar lo que le debe Habitat y renegociar
  con ella (sin descuento y pagando a 60 días como pactaron) o reducir su peso. Si contrata ahora,
  con la caja que tiene y Habitat sin pagar, se arriesga a los números rojos del escenario pesimista.

## 6. Guardar los resultados para Power BI
Se guardan en la base de datos el margen de cada obra, la previsión de caja y el saldo diario,
para usarlos en el dashboard. El cuaderno `06_exportar_powerbi` los exporta a CSV junto con las tablas limpias.

In [ ]:
# Guardar en la base de datos los resultados del análisis que usará el dashboard de Power BI
margen.reset_index().drop(columns=["grupo"], errors="ignore") \
      .to_sql("analisis_margen_obras", bd, if_exists="replace", index=False)
prevision.rename_axis("fecha").reset_index() \
         .melt(id_vars="fecha", var_name="escenario", value_name="saldo_previsto") \
         .to_sql("analisis_prevision_caja", bd, if_exists="replace", index=False)
pd.DataFrame({"fecha": saldo.index, "saldo": saldo.values, "deuda_habitat": deuda_habitat.values}) \
  .to_sql("analisis_saldo_diario", bd, if_exists="replace", index=False)
bd.commit()
sql("SELECT name FROM sqlite_master WHERE name LIKE 'analisis_%'")